In [ ]:
import pandas as pd
import networkx as nx
import numpy as np
import csv
import math
import statistics
import pickle
from itertools import combinations
import os
import heapq
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages
import igraph as ig
from matplotlib.lines import Line2D

# Grafo

In [ ]:
with open('Adulta/colonists_2.5.pickle', 'rb') as f:
            colonists = pickle.load(f)

In [ ]:
colonists_nodes = list(colonists.keys())
colonists_nodes = [str(node) for node in colonists_nodes]

In [ ]:
len(colonists_nodes)

In [ ]:
G = nx.read_graphml("weightedGraph_adult.graphml")

In [ ]:
omologhi = pd.read_csv("../emisferi/adulta/csv/Associazioni.csv", header=0)
omologhi['left_id'] = omologhi['left_id'].astype(str)
omologhi['right_id'] = omologhi['right_id'].astype(str)

In [ ]:
i=0
for n, data in G.nodes(data=True):
    if i%10000==0:
        print(i)
    i+=1
    if data['hemisphere']=='left':
        homolog = omologhi[omologhi['left_id'] == str(n)]['right_id']
        if homolog.empty:
            data['homolog'] = 'not present'
        else:
            data['homolog'] = homolog.iloc[0]
    elif data['hemisphere']=='right':
        homolog = omologhi[omologhi['right_id'] == str(n)]['left_id']
        if homolog.empty:
            data['homolog'] = 'not present'
        else:
            data['homolog'] = homolog.iloc[0]
    else:
        data['homolog'] = 'not present'

In [ ]:
nx.write_graphml(G, "weightedGraph_adulta_homologs.graphml")

In [ ]:
G = nx.read_graphml("weightedGraph_adulta_homologs.graphml")

In [ ]:
G_igraph = ig.Graph.from_networkx(G)

In [ ]:
with open('Adulta/colonists_per_area.pickle', 'rb') as f:
        colonists_per_area = pickle.load(f)

In [ ]:
neurons = pd.read_csv('Data/neurons.csv', header=0)

In [ ]:
neurons

In [ ]:
for n, data in G.nodes(data=True):
    nt_type = neurons[neurons['root_id'] == int(n)]['nt_type']
    if nt_type.empty:
        data['nt_type'] = 'not present'
    else:
        data['nt_type'] = nt_type.iloc[0]

In [ ]:
nx.write_graphml(G, "weightedGraph_adulta_nt.graphml")

In [ ]:
G = nx.read_graphml("weightedGraph_adulta_nt.graphml")

In [ ]:
with open('Adulta/colonists_per_area.pickle', 'rb') as f:
        colonists_per_area = pickle.load(f)

In [ ]:
attributes = [data["superclass"] for _, data in G.nodes(data=True)]

# Conta le occorrenze dei valori degli attributi
attribute_counts = Counter(attributes)

In [ ]:
attribute_counts

# Analisi centralità

In [ ]:
num_20 = G.number_of_nodes()*20/100

In [ ]:
G_igraph.vs[0]

## Betweenness Centrality

In [ ]:
betweenness_all = G_igraph.betweenness(directed=True)

In [ ]:
sorted_nodes = sorted(zip(betweenness_all, G_igraph.vs["_nx_name"]), reverse=True)
sorted_betwenness = {name: value for value, name in sorted_nodes}

In [ ]:
with open('Adulta/betweenness.pickle', 'wb') as f:
    pickle.dump(sorted_betwenness, f)

In [ ]:
with open('Adulta/betweenness.pickle', 'rb') as f:
            sorted_betwenness = pickle.load(f)

In [ ]:
betweenness_colonists = {node: sorted_betwenness[str(node)] for node in colonists_nodes}

In [ ]:
# prendo i nodi con i più alti valori di betweenness (20%)
top_betweenness = heapq.nlargest(int(num_20), sorted_betwenness, key=sorted_betwenness.get)
top_betweenness = [str(node) for node in top_betweenness]

In [ ]:
colonists_betweenness = set(top_betweenness).intersection(colonists_nodes)

In [ ]:
len(colonists_betweenness)

In [ ]:
(len(colonists_betweenness)/len(colonists_nodes))*100

## Closeness Centrality

In [ ]:
closeness_all = G_igraph.closeness()

In [ ]:
sorted_nodes = sorted(zip(closeness_all, G_igraph.vs["_nx_name"]), reverse=True)
sorted_closeness = {name: value for value, name in sorted_nodes}

In [ ]:
with open('Adulta/closeness.pickle', 'wb') as f:
            pickle.dump(sorted_closeness, f)

In [ ]:
with open('Adulta/closeness.pickle', 'rb') as f:
            sorted_closeness = pickle.load(f)

In [ ]:
closeness_colonists = {node: sorted_closeness[str(node)] for node in colonists_nodes}

# prendo i nodi con i più alti valori di betweenness (20%)
num_20 = len(G.nodes())*20/100
top_closeness = heapq.nlargest(int(num_20), sorted_closeness, key=sorted_closeness.get)
top_closeness = [str(node) for node in top_closeness]

colonists_nodes = [str(node) for node in colonists_nodes]
colonists_closeness = set(top_closeness).intersection(colonists_nodes)

len(colonists_closeness)

In [ ]:
(len(colonists_closeness)/len(colonists_nodes))*100

## Degree Centrality

In [ ]:
degree_all = G_igraph.degree()

In [ ]:
sorted_nodes = sorted(zip(degree_all, G_igraph.vs["_nx_name"]), reverse=True)
sorted_degree = {name: value for value, name in sorted_nodes}

In [ ]:
with open('Adulta/degree.pickle', 'wb') as f:
            pickle.dump(sorted_degree, f)

In [ ]:
with open('Adulta/degree.pickle', 'rb') as f:
            sorted_degree = pickle.load(f)

In [ ]:
degree_colonists = {node: sorted_degree[str(node)] for node in colonists_nodes}

# prendo i nodi con i più alti valori di betweenness (20%)
num_20 = len(G.nodes())*20/100
top_degree = heapq.nlargest(int(num_20), sorted_degree, key=sorted_degree.get)
top_degree = [str(node) for node in top_degree]

colonists_degree = set(top_degree).intersection(colonists_nodes)

len(colonists_degree)

In [ ]:
(len(colonists_degree)/len(colonists_nodes))*100

In [ ]:
top_3 = set(colonists_betweenness).intersection(colonists_closeness).intersection(colonists_degree)

In [ ]:
print('Out of ' + str(G.number_of_nodes()) + ' neurons ' + str(len(colonists_nodes)) + ' (' + str(round(len(colonists_nodes)/G.number_of_nodes()*100, 2)) + ' %) are colonist neurons' )
print('and out of ' + str(len(colonists_nodes)) + ' colonists ' + str(len(colonists_betweenness)) + ' (' + str(round(len(colonists_betweenness)/len(colonists_nodes)*100, 2)) +  '%) of them belong to the top 20% of betweenness' )
print('and out of ' + str(len(colonists_nodes)) + ' colonists ' + str(len(colonists_closeness)) + ' (' + str(round(len(colonists_closeness)/len(colonists_nodes)*100, 2)) + '%) of them belong to the top 20% of closeness' )
print('and out of ' + str(len(colonists_nodes)) + ' colonists ' + str(len(colonists_degree)) + ' (' + str(round(len(colonists_degree)/len(colonists_nodes)*100, 2)) +  '%) of them belong to the top 20% of degree' )
print('and out of ' + str(len(colonists_nodes)) + ' colonists ' + str(len(top_3)) + ' (' + str(round(len(top_3)/len(colonists_nodes)*100, 2)) +  '%) of them belong to the intersection of the three of them' )
print()

## Centralità per aree

In [ ]:
superclasses = list({attr.get('superclass', 'unknown') for _, attr in G.nodes(data=True)})

In [ ]:
colonists_per_area =  {key: [] for key in superclasses}

for colonist in colonists_nodes:
    area = G.nodes[colonist]['superclass']
    colonists_per_area[area].append(colonist) 

In [ ]:
with open('Adulta/colonists_per_area.pickle', 'wb') as f:
        pickle.dump(colonists_per_area, f)

In [ ]:
for area in superclasses:
    nodes = [nodo for nodo, attr in G.nodes(data=True) if attr.get('superclass') == area]
    area_colonists = colonists_per_area[area] 
    
    colonists_betweenness = set(top_betweenness).intersection(area_colonists)
    colonists_closeness = set(top_closeness).intersection(area_colonists)
    colonists_degree = set(top_degree).intersection(area_colonists)
    top_3 = colonists_betweenness.intersection(colonists_closeness).intersection(colonists_degree)

    if len(area_colonists)!=0:
        print(area + ': \nout of ' + str(len(nodes)) + ' neurons ' + str(len(area_colonists)) + ' (' + str(round(len(area_colonists)/len(nodes)*100, 2)) + ' %) are colonist neurons' )
        print('and out of ' + str(len(area_colonists)) + ' colonists ' + str(len(colonists_betweenness)) + ' (' + str(round(len(colonists_betweenness)/len(area_colonists)*100, 2)) +  '%) of them belong to the top 20% of betweenness' )
        print('and out of ' + str(len(area_colonists)) + ' colonists ' + str(len(colonists_closeness)) + ' (' + str(round(len(colonists_closeness)/len(area_colonists)*100, 2)) + '%) of them belong to the top 20% of closeness' )
        print('and out of ' + str(len(area_colonists)) + ' colonists ' + str(len(colonists_degree)) + ' (' + str(round(len(colonists_degree)/len(area_colonists)*100, 2)) +  '%) of them belong to the top 20% of degree' )
        print('and out of ' + str(len(area_colonists)) + ' colonists ' + str(len(top_3)) + ' (' + str(round(len(top_3)/len(area_colonists)*100, 2)) +  '%) of them belong to the intersection of the three of them' )
        print()
    else:
        print(area + ': \nout of ' + str(len(nodes)) + ' neurons ' + str(len(area_colonists)) + ' (' + str(round(len(area_colonists)/len(nodes)*100, 2)) + ' %) are colonist neurons' )
        print()

## Eccentricity

In [ ]:
sccs = list(nx.strongly_connected_components(G))
largest_scc = max(sccs, key=len)
largest_scc_subgraph = G.subgraph(largest_scc).copy()

In [ ]:
eccentricity_all = nx.eccentricity(largest_scc_subgraph)

In [ ]:
with open('Adulta/eccentricity_all.pickle', 'wb') as f:
            pickle.dump(eccentricity_all, f)

In [ ]:
with open('Adulta/eccentricity_all.pickle', 'rb') as f:
            eccentricity_all = pickle.load(f)

In [ ]:
eccentrity_colonist = {node: eccentricity_all.get(str(node), 0) for node in colonists_nodes}

In [ ]:
# prendo i nodi con i più alti valori di betweenness (20%)
num_20 = len(G.nodes())*20/100
top_eccentricity = heapq.nlargest(int(num_20), eccentricity_all, key=eccentricity_all.get)
top_eccentricity = [str(node) for node in top_eccentricity]

colonists_eccentricity = set(top_eccentricity).intersection(colonists_nodes)

len(colonists_eccentricity)

In [ ]:
(len(colonists_eccentricity)/len(colonists_nodes))*100

In [ ]:
with open('Adulta/colonists_per_area.pickle', 'rb') as f:
            colonists_per_area = pickle.load(f)

In [ ]:
for area in superclasses:
    nodes = [nodo for nodo, attr in G.nodes(data=True) if attr.get('superclass') == area]
    area_colonists = colonists_per_area[area] 
    
    colonists_eccentricity = set(top_eccentricity).intersection(area_colonists)

    if len(area_colonists)!=0:
        print(area + ': \nout of ' + str(len(nodes)) + ' neurons ' + str(len(area_colonists)) + ' (' + str(round(len(area_colonists)/len(nodes)*100, 2)) + ' %) are colonist neurons' )
        print('and out of ' + str(len(area_colonists)) + ' colonists ' + str(len(colonists_eccentricity)) + ' (' + str(round(len(colonists_eccentricity)/len(area_colonists)*100, 2)) +  '%) of them belong to the top 20% of eccentricity' )
        print()
    else:
        print(area + ': \nout of ' + str(len(nodes)) + ' neurons ' + str(len(area_colonists)) + ' (' + str(round(len(area_colonists)/len(nodes)*100, 2)) + ' %) are colonist neurons' )
        print()

# Caratterizzazione

## Singolo vs a raggiera

In [ ]:
colonists_type = {'singolo':[], 'a raggiera':[]}

for colonist, clusters in colonists.items():
    if len(clusters)==1:
        colonists_type['singolo'].append(colonist)
    else:
        colonists_type['a raggiera'].append(colonist)

In [ ]:
len(colonists_type['singolo'])

In [ ]:
len(colonists_type['singolo'])/len(colonists_nodes)

In [ ]:
len(colonists_type['a raggiera'])

In [ ]:
len(colonists_type['a raggiera'])/len(colonists_nodes)

## Analisi archi lunghi/archi corti

In [ ]:
with open('Adulta/dist_info_superclasses.pickle', 'rb') as f:
    dist_info_superclasses = pickle.load(f)

In [ ]:
for celltype, info in dist_info_superclasses.items():
    print(celltype + ' average distance: ' + str(round(info['average'],4)))

In [ ]:
long_short_ratio = []
long_short_ratio_tuple = []

for colonist in colonists_nodes:
    superclass = G.nodes[colonist]['superclass']
    out_edges = list(G.out_edges(colonist, data=True))
    num_long = 0
    for edge in out_edges:
        if edge[2]['weight'] > 2.2*dist_info_superclasses[superclass]['average']:
            num_long+=1
    long_short_ratio.append(num_long/len(out_edges))
    long_short_ratio_tuple.append((colonist, num_long/len(out_edges)))

In [ ]:
def plot_ratio_dist(data):
    plt.figure(figsize=(8, 6))
    sns.histplot(data, kde=True, bins=20,binrange=(0,1))
    #plt.title("All colonists distribution of the ratio of long vs all connections - adult", fontsize=10)
    plt.xlabel("Value")
    plt.ylabel("Number of colonists")
    plt.tight_layout()
    plt.savefig("AdultaImg/adulta_distribution_ratio_connections.pdf")
    plt.show()

In [ ]:
plot_ratio_dist(long_short_ratio)

In [ ]:
len(long_short_ratio)

In [ ]:
longest = []

for (c, r) in long_short_ratio_tuple:
    if r > 0.95:
        longest.append((c,r))

In [ ]:
longest_superclass = []

for (c, r) in longest:
    areas = superclass = G.nodes[c]['superclass']
    longest_superclass.append(areas)

In [ ]:
out_edges_length = []
out_edges_long_length = []

for (colonist, r) in longest:
    superclass = G.nodes[colonist]['superclass']
    out_edges = list(G.out_edges(colonist, data=True))
    num_long = 0
    for edge in out_edges:
        if edge[2]['weight'] > 2.2*dist_info_superclasses[superclass]['average']:
            num_long+=1
    out_edges_long_length.append(num_long)
    out_edges_length.append(len(out_edges))

In [ ]:
sum(out_edges_length)/len(out_edges_length)

In [ ]:
sum(out_edges_long_length)/len(out_edges_long_length)

In [ ]:
with open('Adulta/last_bin.pickle', 'wb') as f:
    pickle.dump(out_edges_length, f)

In [ ]:
longest_nodes = set()

for (c,r) in longest:
    longest_nodes.add(c)

len(longest_nodes.intersection(colonists_type['a raggiera']))
len(longest_nodes.intersection(colonists_type['singolo']))

## Distribuzione archi lunghi/corti per area

In [ ]:
import matplotlib.ticker as mticker

In [ ]:
colors_dict = {
    'sensory': '#FFB6C1',
    'ascending': '#FF6347',
    'central': "#FFFE02",
    'optic': "#6495ED",
    'visual_projection': "#D2B48C",
    'visual_centrifugal': "#DA70D6",
    'descending': "#F4A460",
    'motor': "#87CEFA",
    'endocrine': '#90EE90'
}

In [ ]:
with open('Adulta/colonists_per_area.pickle', 'rb') as f:
            colonists_per_area = pickle.load(f)

In [ ]:
def plot_ratio_dist(dati, nomi_aree, colors, titolo, path_pdf):
    """
    Genera un histplot per ogni set di dati passato e li salva in un file PDF.
    Al massimo 6 grafici per foglio.

    Parametri:
    - dati: Lista di liste o array. Ogni elemento della lista è un set di dati.
    - titolo: Titolo generale del PDF.
    - path_pdf: Percorso del file PDF in cui salvare i grafici.
    """
    num_subplot = len(dati)  # Numero di set di dati
    grafici_per_foglio = 6  # Numero massimo di grafici per pagina
    num_pagine = (num_subplot + grafici_per_foglio - 1) // grafici_per_foglio  # Calcolo numero di pagine
    
    # Creare un PDF per salvare i grafici
    with PdfPages(path_pdf) as pdf:
        for pagina in range(num_pagine):
            # Calcolare i sottogruppi da visualizzare in questa pagina
            start_idx = pagina * grafici_per_foglio
            end_idx = min((pagina + 1) * grafici_per_foglio, num_subplot)
            dati_pagina = dati[start_idx:end_idx]
            
            # Creazione del layout per la pagina corrente
            fig, axes = plt.subplots(3, 2, figsize=(10, 9), constrained_layout=True)  # 2 righe e 3 colonne
            axes = axes.flatten()  # Appiattire l'array per facilitare la gestione
            # plt.subplots_adjust(wspace=0.5, hspace=0.4)

            
            for idx, (ax, dataset, color) in enumerate(zip(axes, dati_pagina, colors)):
                global_idx = start_idx + idx
                sns.histplot(dataset, kde=True, ax=ax, bins=20, binrange=(0,1), color=color)
                ax.set_title(f'{nomi_aree[global_idx]}')
                ax.set_xlabel('Value')
                ax.set_ylabel('Number of colonists')

                print(nomi_aree[global_idx])
                
                # Calcolo della frequenza massima basata sull'istogramma
                counts, _ = np.histogram(dataset, bins=20, range=(0, 1))  # Calcola la frequenza dei bin
                max_freq = max(counts)  # Trova la frequenza massima
                step = max(1, max_freq // 5)  # Calcola il passo dei tick (massimo 5 tick)
            
                # Imposta i tick dell'asse y con valori interi e adeguati
                ax.yaxis.set_major_locator(mticker.MultipleLocator(step))
            
            # Rimuovere i subplot vuoti se ce ne sono
            for ax in axes[len(dati_pagina):]:
                ax.axis('off')

            plt.tight_layout()
            
            # Titolo generale della pagina
            # fig.suptitle(f'{titolo}', fontsize=16)
            
            # Salvare la figura corrente nel PDF
            pdf.savefig(fig)
            plt.close(fig)

In [ ]:
aree = list({attr.get('superclass', 'unknown') for _, attr in G.nodes(data=True)})

In [ ]:
long_short_ratio_aree = {}
long_short_ratio_aree_nodes = {}

for area in aree:
    colonists_area = colonists_per_area[area]
    long_short_ratio_aree[area] = []
    long_short_ratio_aree_nodes[area] = []
    for colonist in colonists_area:
        celltype = G.nodes[colonist]['superclass']
        out_edges = list(G.out_edges(colonist, data=True))
        num_long = 0
        for edge in out_edges:
            if edge[2]['weight'] > 2.2*dist_info_superclasses[celltype]['average']:
                num_long+=1
        long_short_ratio_aree[area].append(num_long/len(out_edges))
        long_short_ratio_aree_nodes[area].append((colonist, num_long/len(out_edges)))
    if long_short_ratio_aree[area]==[]:
        long_short_ratio_aree.pop(area)

In [ ]:
len(aree)

In [ ]:
long_short_ratio_aree_dati = list(long_short_ratio_aree.values())
aree_nomi = list(long_short_ratio_aree.keys())

colors_dict_copy = colors_dict.copy()
for elem in colors_dict:
    if elem not in aree_nomi:
        colors_dict_copy.pop(elem)
        
colors_dict_copy = dict(sorted(colors_dict_copy.items(), key=lambda pair: aree_nomi.index(pair[0])))
colors = list(colors_dict_copy.values())

In [ ]:
plot_ratio_dist(long_short_ratio_aree_dati, aree_nomi, colors, "Distribution per area", "AdultaImg/adulta_distribution_ratio_connections_nuova_versione.pdf")

In [ ]:
visual_projection_data = long_short_ratio_aree_dati[0]

In [ ]:
plt.figure(figsize=(8, 6))
fig, ax = plt.subplots()  # Anche un singolo plot usa fig e ax
sns.histplot(visual_projection_data, kde=True, bins=20, ax=ax, binrange=(0,1), color='#D2B48C')
# Calcolo della frequenza massima basata sull'istogramma
counts, _ = np.histogram(visual_projection_data, bins=20, range=(0, 1))  # Calcola la frequenza dei bin
max_freq = max(counts)  # Trova la frequenza massima
step = max(1, max_freq // 5)  # Calcola il passo dei tick (massimo 5 tick)

# Imposta i tick dell'asse y con valori interi e adeguati
ax.yaxis.set_major_locator(mticker.MultipleLocator(step))
plt.xlabel("Value")
plt.ylabel("Number of colonists")
plt.tight_layout()
plt.savefig("AdultaImg/adulta_distribution_ratio_connections_visual_projection.pdf")
plt.show()

In [ ]:
long_colonist_optic = []

for (colonist, ratio) in long_short_ratio_aree_nodes['optic']:
    if ratio>0.5:
        long_colonist_optic.append(colonist)

In [ ]:
len(long_short_ratio_aree_nodes['optic'])

In [ ]:
len(long_colonist_optic)

In [ ]:
len(set(colonists_type['a raggiera']).intersection(long_colonist_optic))

In [ ]:
long_colonist_sensory = []

for (colonist, ratio) in long_short_ratio_aree_nodes['sensory']:
    if ratio>0.5:
        long_colonist_sensory.append(colonist)

In [ ]:
len(long_short_ratio_aree_nodes['sensory'])

In [ ]:
len(set(colonists_type['a raggiera']).intersection(long_colonist_sensory))

In [ ]:
long_colonist_central = []

for (colonist, ratio) in long_short_ratio_aree_nodes['central']:
    if ratio>0.5:
        long_colonist_central.append(colonist)

In [ ]:
len(long_short_ratio_aree_nodes['central'])

In [ ]:
len(set(colonists_type['a raggiera']).intersection(long_colonist_central))

# Distribuzione delle connessioni per area

In [ ]:
colors_dict = {
    'sensory': '#FFB6C1',
    'ascending': '#FF6347',
    'central': "#FFFE02",
    'optic': "#6495ED",
    'visual_projection': "#D2B48C",
    'visual_centrifugal': "#DA70D6",
    'descending': "#F4A460",
    'motor': "#87CEFA",
    'endocrine': '#90EE90'
}

In [ ]:
connections_per_area = {}

for area in superclasses:
    colonists_area = colonists_per_area[area]
    vicini = []
    for colonist in colonists_area:
        succ = list(G.successors(colonist))
        pred = list(G.predecessors(colonist))
        vicini.extend(succ)
        vicini.extend(pred)
        break

    celltypes = []
    for vicino in vicini: 
        celltype = G.nodes[vicino]['superclass']
        celltypes.append(celltype)

    connections_per_area[area] = celltypes
    

In [ ]:
# Trasformare i dati in un formato tabellare (adatto per Seaborn)
rows = []
for area, neighbors in connections_per_area.items():
    counts = Counter(neighbors)
    for neighbor, count in counts.items():
        rows.append({"Area": area, "Neighbor": neighbor, "Count": count})

df = pd.DataFrame(rows)

# Configurazione per salvare i grafici nel PDF
with PdfPages("AdultaImg/areas_connections_with_relative_distribution.pdf") as pdf:
    plots_per_page = 6  # Numero totale di grafici per pagina
    rows_per_page = 3  # Numero di righe per pagina
    cols_per_page = 2  # Numero di colonne per pagina
    fig, axes = plt.subplots(rows_per_page, cols_per_page, figsize=(12, 18))  # Griglia 3x2

    # Ciclo sulle aree
    for i, area in enumerate(df["Area"].unique()):
        subset = df[df["Area"] == area]  
        subset = subset.sort_values(by="Count", ascending=False)  # Ordinare i dati

        # Calcolo della distribuzione relativa
        relative_distribution = subset["Count"] / subset["Count"].sum()

        # Identificare quale subplot usare
        row, col = divmod(i % plots_per_page, cols_per_page)
        ax = axes[row][col]
        sns.barplot(data=subset, x="Neighbor", y="Count", palette=colors_dict, edgecolor="black", ax=ax, hue="Neighbor")

        # Impostare titoli ed etichette
        ax.set_title(f"Colonists connections - {area}")
        ax.set_xlabel("Areas")
        ax.set_ylabel("Number of connections")
        ax.set_xticks(range(len(subset["Neighbor"])))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

        # Se è l'ultimo grafico della pagina o l'ultimo in assoluto, salvare e creare una nuova figura
        if (i + 1) % plots_per_page == 0 or i == len(df["Area"].unique()) - 1:
            if i == len(df["Area"].unique()) - 1:
                row, col = divmod((i+1) % plots_per_page, cols_per_page)
                for i in range(row, rows_per_page):  # partendo dalla riga 1 (seconda riga)
                    for j in range(col, cols_per_page):  # partendo dalla colonna 1 (seconda colonna)
                        axes[i, j].axis('off')  # Nascondi gli assi
                for i in range(row+1, rows_per_page):  # partendo dalla riga 1 (seconda riga)
                    for j in range(0, cols_per_page):  # partendo dalla colonna 1 (seconda colonna)
                        axes[i, j].axis('off')
            plt.tight_layout()
            pdf.savefig(fig)  # Salvare la pagina nel PDF
            plt.close(fig)  # Chiudere la figura corrente

            # Preparare una nuova pagina se ci sono altri grafici
            if i != len(df["Area"].unique()) - 1:
                fig, axes = plt.subplots(rows_per_page, cols_per_page, figsize=(12, 18))

# Distribuzione dei coloni per emisfero

In [ ]:
superclasses = list({attr.get('superclass', 'unknown') for _, attr in G.nodes(data=True)})

In [ ]:
colonists_per_emisfero = {'left':[], 'right':[], 'not present':[]}

for colonist in colonists_nodes:
    hemisphere = G.nodes[colonist]['hemisphere']
    if hemisphere != hemisphere:
        colonists_per_emisfero['not present'].append(colonist)
    else:
        colonists_per_emisfero[hemisphere].append(colonist)

In [ ]:
colonists_per_emisfero

In [ ]:
print('Left hemisphere: ' + str(len(colonists_per_emisfero['left'])))
print('Right hemisphere: ' + str(len(colonists_per_emisfero['right'])))

In [ ]:
colonists_homologs_left = {'y':[], 'n':[]}
colonists_homologs_right = {'y':[], 'n':[]}

for colonist in colonists_per_emisfero['left']:
    homolog = G.nodes[colonist]['homolog']
    if homolog == 'not present':
        colonists_homologs_left['n'].append(colonist)
    else:
        colonists_homologs_left['y'].append(colonist)

for colonist in colonists_per_emisfero['right']:
    homolog = G.nodes[colonist]['homolog']
    if homolog == 'not present':
        colonists_homologs_right['n'].append(colonist)
    else:
        colonists_homologs_right['y'].append(colonist)

In [ ]:
print('Colonists without a homolog - left: ' + str(len(colonists_homologs_left['n'])))
print('Colonists with a homolog - left: ' + str(len(colonists_homologs_left['y'])))
print('Colonists without a homolog - right: ' + str(len(colonists_homologs_right['n'])))
print('Colonists with a homolog - right: ' + str(len(colonists_homologs_right['y'])))

In [ ]:
# y contiene la lista dei colonist il cui omologo è colonist
# n contiene la lista dei colonist il cui omologo non è colonist
colonists_homologs_colonists_left = {'y': [], 'n': []}
colonists_homologs_colonists_right = {'y': [], 'n': []}

homologsnotcolonists_of_left_colonists = []
homologsnotcolonists_of_right_colonists = []

for colonist in colonists_homologs_left['y']:
    homolog = G.nodes[colonist]['homolog']
    if homolog in colonists_nodes:
        colonists_homologs_colonists_left['y'].append(colonist)
    else:
        colonists_homologs_colonists_left['n'].append(colonist)
        homologsnotcolonists_of_left_colonists.append(homolog)

for colonist in colonists_homologs_right['y']:
    homolog = G.nodes[colonist]['homolog']
    if homolog in colonists_nodes:
        colonists_homologs_colonists_right['y'].append(colonist)
    else:
        colonists_homologs_colonists_right['n'].append(colonist)
        homologsnotcolonists_of_right_colonists.append(homolog)

In [ ]:
print('Colonists whose homolog is a colonist - left: ' + str(len(colonists_homologs_colonists_left['y'])))
print('Colonists whose homolog is not a colonist - left: ' + str(len(colonists_homologs_colonists_left['n'])))
print('Colonists whose homolog is a colonist - right: ' + str(len(colonists_homologs_colonists_right['y'])))
print('Colonists whose homolog is not a colonist - right: ' + str(len(colonists_homologs_colonists_right['n'])))

In [ ]:
left_colonists_homologs_not_colonists = colonists_homologs_colonists_left['n']
right_colonists_homologs_not_colonists = colonists_homologs_colonists_right['n']

### Aree dei colonist il cui omologo non è colonist

In [ ]:
superclasses_left_homolognocolonist = []
for colonist in left_colonists_homologs_not_colonists:
    superclass = G.nodes[colonist]['superclass']
    superclasses_left_homolognocolonist.append(superclass)

superclasses_right_homolognocolonist = []
for colonist in right_colonists_homologs_not_colonists:
    superclass = G.nodes[colonist]['superclass']
    superclasses_right_homolognocolonist.append(superclass)

In [ ]:
superclasses_left_nohomolog_counts = {}

for superclass in superclasses:
    superclasses_left_nohomolog_counts[superclass] = superclasses_left_homolognocolonist.count(superclass)

superclasses_right_nohomolog_counts = {}

for superclass in superclasses:
    superclasses_right_nohomolog_counts[superclass] = superclasses_right_homolognocolonist.count(superclass)

In [ ]:
superclasses_left_nohomolog_counts

In [ ]:
superclasses_right_nohomolog_counts

### Gli omologhi non colonist dei colonist sono poco sotto soglia? 

In [ ]:
with open('Adulta/dist_info_superclasses.pickle', 'rb') as f:
    dist_info_superclasses = pickle.load(f)

In [ ]:
G.number_of_nodes()

In [ ]:
p=2

In [ ]:
candidate_colonists_leftc_rightnc = []
candidate_colonists_rightc_leftnc = []
not_present = []

for colonist in left_colonists_homologs_not_colonists:
    homolog = G.nodes[colonist]['homolog']
    if G.has_node(homolog):
        superclass = G.nodes[colonist]['superclass']
        info_superclass = dist_info_superclasses[superclass]
        out_edges = list(G.out_edges(homolog, data=True))
        for edge in out_edges:
            if edge[2]['weight'] > p*info_superclass['average']:
                candidate_colonists_leftc_rightnc.append(homolog)
                break
    else:
        not_present.append(homolog)

for colonist in right_colonists_homologs_not_colonists:
    homolog = G.nodes[colonist]['homolog']
    if G.has_node(homolog):
        superclass = G.nodes[homolog]['superclass']
        info_superclass = dist_info_superclasses[superclass]
        out_edges = list(G.out_edges(homolog, data=True))
        for edge in out_edges:
            if edge[2]['weight'] > p*info_superclass['average']:
                candidate_colonists_rightc_leftnc.append(homolog)
                break
    else:
        not_present.append(homolog)

In [ ]:
print(len(candidate_colonists_leftc_rightnc))
print(len(candidate_colonists_rightc_leftnc))

In [ ]:
print(len(not_present))

In [ ]:
target_nodes_dict_leftc_rightnc = {}
target_nodes_dict_rightc_leftnc = {}
num_target_nodes_1 = {}
num_target_nodes_2 = {}
        
for source in candidate_colonists_leftc_rightnc:
    superclass = G.nodes[source]['superclass']
    info_superclass = dist_info_superclasses[superclass]
    out_edges = list(G.out_edges(source, data=True))
    target_nodes = set()
    for edge in out_edges:
        if edge[2]['weight'] > p*info_superclass['average']:
            target_nodes.add(edge[1])
    if len(target_nodes)>1:
        target_nodes_dict_leftc_rightnc[source] = list(target_nodes)
    num_target_nodes_1[source] = len(target_nodes)

for source in candidate_colonists_rightc_leftnc:
    superclass = G.nodes[source]['superclass']
    info_superclass = dist_info_superclasses[superclass]
    out_edges = list(G.out_edges(source, data=True))
    target_nodes = set()
    for edge in out_edges:
        if edge[2]['weight'] > p*info_superclass['average']:
            target_nodes.add(edge[1])
    if len(target_nodes)>1:
        target_nodes_dict_rightc_leftnc[source] = list(target_nodes)
    num_target_nodes_2[source] = len(target_nodes)

In [ ]:
len(target_nodes_dict_leftc_rightnc)

In [ ]:
target_nodes_dict_rightc_leftnc

In [ ]:
len(target_nodes_dict_rightc_leftnc)

In [ ]:
tot_dist = list(nx.get_edge_attributes(G, 'weight').values())
avg_dist = sum(tot_dist) / len(tot_dist)
avg_dist

In [ ]:
verified_colonists = {} 

for candidate, targets in target_nodes_dict_leftc_rightnc.items():
    targets_info = {nodo: G.nodes[nodo] for nodo in targets}
    
    subgraph = nx.Graph()
    subgraph.add_nodes_from(list(targets))
    
    edges = list(combinations(list(targets), 2))
    subgraph.add_edges_from(edges)
    w = {}
    for source, target, data in subgraph.edges(data=True):
        source_x = targets_info[source]['x']
        source_y = targets_info[source]['y']
        source_z = targets_info[source]['z']
        target_x = targets_info[target]['x']
        target_y = targets_info[target]['y']
        target_z = targets_info[target]['z']
        distanza = math.sqrt((source_x - target_x)**2 + (source_y - target_y)**2 + (source_z - target_z)**2)
        data['weight'] = distanza
    to_remove = []
    for source, target, data in subgraph.edges(data=True):
        if data['weight']>avg_dist:
            to_remove.append((source, target))
    
    subgraph.remove_edges_from(to_remove)

    prov_connected_components = list(nx.connected_components(subgraph))
    connected_components = []
    print(prov_connected_components)
    
    for connected_component in prov_connected_components:
        if len(connected_component) > 1:
            connected_components.append(connected_component)

    if len(connected_components) >= 1:
        verified_colonists[candidate] = connected_components

In [ ]:
len(verified_colonists)

In [ ]:
verified_colonists = {} 

for candidate, targets in target_nodes_dict_rightc_leftnc.items():
    targets_info = {nodo: G.nodes[nodo] for nodo in targets}
    
    subgraph = nx.Graph()
    subgraph.add_nodes_from(list(targets))
    
    edges = list(combinations(list(targets), 2))
    subgraph.add_edges_from(edges)
    w = {}
    for source, target, data in subgraph.edges(data=True):
        source_x = targets_info[source]['x']
        source_y = targets_info[source]['y']
        source_z = targets_info[source]['z']
        target_x = targets_info[target]['x']
        target_y = targets_info[target]['y']
        target_z = targets_info[target]['z']
        distanza = math.sqrt((source_x - target_x)**2 + (source_y - target_y)**2 + (source_z - target_z)**2)
        data['weight'] = distanza
    to_remove = []
    for source, target, data in subgraph.edges(data=True):
        if data['weight']>avg_dist:
            to_remove.append((source, target))
    
    subgraph.remove_edges_from(to_remove)

    prov_connected_components = list(nx.connected_components(subgraph))
    connected_components = []
    
    for connected_component in prov_connected_components:
        if len(connected_component) > 1:
            connected_components.append(connected_component)

    if len(connected_components) >= 1:
        verified_colonists[candidate] = connected_components

In [ ]:
len(verified_colonists)

# Distribuzione dei colonist neuron per area ed emisfero

In [ ]:
superclasses = ['sensory', 'ascending', 'central', 'optic', 'visual_projection', 'visual_centrifugal', 'descending', 'motor', 'endocrine']

In [ ]:
colors_dict = {
    'sensory': '#FFB6C1',
    'ascending': '#FF6347',
    'central': "#FFFE02",
    'optic': "#6495ED",
    'visual_projection': "#D2B48C",
    'visual_centrifugal': "#DA70D6",
    'descending': "#F4A460",
    'motor': "#87CEFA",
    'endocrine': '#90EE90'
}

In [ ]:
colonists_areas = {'area': [], 'sinistro': [], 'destro':[], 'sinistro_percentuale':[], 'destro_percentuale':[]}

for area in superclasses:
    nodes_left = [nodo for nodo, attr in G.nodes(data=True) if attr.get('superclass') == area and attr.get('hemisphere') == 'left']
    nodes_right = [nodo for nodo, attr in G.nodes(data=True) if attr.get('superclass') == area and attr.get('hemisphere') == 'right']
    tot_nodes_area = len(nodes_left) + len(nodes_right)
    colonists_left = set(nodes_left).intersection(colonists_nodes)
    colonists_right = set(nodes_right).intersection(colonists_nodes)

    colonists_areas['area'].append(area)
    colonists_areas['sinistro'].append(len(colonists_left))
    colonists_areas['destro'].append(len(colonists_right))
    colonists_areas['sinistro_percentuale'].append(len(colonists_left)/tot_nodes_area*100)
    colonists_areas['destro_percentuale'].append(len(colonists_right)/tot_nodes_area*100)

colonists_areas_df = pd.DataFrame.from_dict(colonists_areas)

In [ ]:
# Bar width
bar_width = 0.8

# Apply colors based on the custom order
colors = [colors_dict[area] for area in colonists_areas_df['area']]

# Positions of the bars on the y-axis
y_positions = np.arange(len(colonists_areas_df['area']))[::-1]

fig = plt.figure(figsize=(15, 10))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 0.2, 1])

# First chart - left
ax1 = fig.add_subplot(gs[0, 0])
bars1 = plt.barh(y_positions + bar_width, colonists_areas_df['sinistro'], bar_width, label='Left hemisphere', color=colors)
ax1.invert_xaxis()
ax1.set_xlim(0, 2500)
ax1.set_xlabel('Number of Colonists', fontsize=14)
ax1.tick_params(left=False)
ax1.yaxis.set_ticks([])
ax1.invert_xaxis()
ax1.set_title("Left Hemisphere")

for bar, percent in zip(bars1, colonists_areas_df['sinistro_percentuale']):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, "%.2f" % percent + "%", va='center', ha='right', fontsize=10, color='black')
# Middle labels
for i, area in enumerate(colonists_areas_df['area']):
    plt.text(0.5, (y_positions[i] + bar_width) / len(y_positions), area, va='center', ha='center', fontsize=10, transform=fig.transFigure)


# Second chart - right
ax3 = fig.add_subplot(gs[0, 2])
bars2 = ax3.barh(y_positions + bar_width, colonists_areas_df['destro'], bar_width, label='Left hemisphere', color=colors)
ax3.set_xlim(0, 2500)
ax3.set_xlabel('Number of Colonists', fontsize=14)
ax3.tick_params(left=False)
ax3.yaxis.set_ticks([])
ax3.set_title("Right Hemisphere")

for bar, percent in zip(bars2, colonists_areas_df['destro_percentuale']):
    ax3.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, "%.2f" % percent + "%", va='center', ha='left', fontsize=10, color='black')

#plt.tight_layout()
plt.savefig('AdultaImg/aree_emisferi_colonist_adulta.png')
plt.show()


# Analisi tipologia neurotrasmettitore

In [ ]:
ntransmitters = list({attr.get('nt_type', 'not present') for _, attr in G.nodes(data=True)})
ntransmitters = [x for x in ntransmitters if str(x) != 'nan']

In [ ]:
ntransmitters

## Conteggio tipo neurotrasmettitore

In [ ]:
types_nt_dict_count = {key: 0 for key in ntransmitters}
types_nt_dict_count['N/A'] = 0

for node in G.nodes():
        n_nt_type = G.nodes[node].get('nt')
        if n_nt_type and not (isinstance(n_nt_type, float) and math.isnan(n_nt_type)):
            types_nt_dict_count[n_nt_type] += 1
        else:
            types_nt_dict_count['N/A'] += 1


In [ ]:
types_nt_dict_count.pop('N/A', None)
types_nt_dict_count

In [ ]:
types_nt_dict_count['SER']/sum(types_nt_dict_count.values())*100

In [ ]:
types_nt_dict_count = {key: 0 for key in ntransmitters}
types_nt_dict_count['N/A'] = 0

for node in colonists_nodes:
        n_nt_type = G.nodes[node].get('nt')
        if n_nt_type and not (isinstance(n_nt_type, float) and math.isnan(n_nt_type)):
            types_nt_dict_count[n_nt_type] += 1
        else:
            types_nt_dict_count['N/A'] += 1

In [ ]:
types_nt_dict_count.pop('N/A', None)
types_nt_dict_count

In [ ]:
types_nt_dict_count['SER']/sum(types_nt_dict_count.values())*100

In [ ]:
aree = list({attr.get('superclass', 'unknown') for _, attr in G.nodes(data=True)})

In [ ]:
nt_dict_colors = {
    'ACH': '#AEC6CF',
    'DA': '#A8D5BA',
    'SER': '#FFDAB9',
    'OCT': '#FFFACD',
    'GABA': '#E6E6FA',
    'GLUT': '#D3D3D3',
    'N/A': 'grey'
}

nt_dict_colors = {
    'ACH': '#0072B2',
    'DA': '#D55E00',
    'SER': '#009E73',
    'OCT': '#CC79A7',
    'GABA': '#D81B60',
    'GLUT': '#F0E442',
    'N/A': '#333333'
}

In [ ]:
def grafico_a_barre_multi(area, types, types_dict_count, count, title, path, all_neurons):
    """
    Genera più subplot, ciascuno contenente un barplot, organizzato in un layout flessibile.
    
    Parametri:
    - types: Lista delle types (una lista di liste, ogni lista corrisponde a un subplot)
    - count: Lista dei valori di frequenza (una lista di liste, ogni lista corrisponde a un subplot)
    - title: Titolo generale del grafico
    - path: Percorso del file per il salvataggio del grafico
    """
    num_subplot = len(types)  # Numero di subplot
    rows = (num_subplot + 1) // 2  # Numero di righe (due subplot per riga)
    cols = 2 if num_subplot > 1 else 1  # Numero di colonne
    
    # Creazione del layout dei subplot
    fig, axes = plt.subplots(rows, cols, figsize=(10, 5 * rows), constrained_layout=True)
    if num_subplot == 1:
        axes = [axes]  # Se c'è un solo subplot, convertirlo in una lista

    axes = axes.flatten()  # Appiattire gli assi per iterare facilmente
    
    for idx, (ax, t, freq) in enumerate(zip(axes, types, count)):
        # Ordinare i dati
        dati = list(zip(t, freq))
        dati_ordinati = sorted(dati, key=lambda x: x[1], reverse=True)
        sort_types = [x[0] for x in dati_ordinati]
        sort_count = [x[1] for x in dati_ordinati]
        sort_perc = []
        for nt, t in enumerate(sort_types):
            if types_dict_count[area[idx]][t]!=0:
                sort_perc.append(sort_count[nt]/types_dict_count[area[idx]][t]*100)
            else:
                sort_perc.append(0)
        sort_colori = [nt_dict_colors[x[0]] for x in dati_ordinati]
        #if all_neurons:
        if True:
            sort_perc = [x / sum(sort_count)*100 for x in sort_count]
        
        # Calcolo della posizione delle barre
        larghezza_barra = 0.5
        spazio_vuoto = 0.5
        larghezza_totale = larghezza_barra + spazio_vuoto
        posizione_barre = [i * larghezza_totale for i in range(len(sort_count))]
        
        # Creazione del barplot nel subplot corrente
        ax.bar(posizione_barre, sort_count, width=larghezza_barra, color=sort_colori)
        ax.set_xticks(posizione_barre)
        ax.set_xticklabels(sort_types, ha='right')
        ax.set_xlabel('Type')
        ax.set_ylabel('Frequency')
        ax.set_title(f'{area[idx]}')
        ax.margins(y=0.1)
        
        print(area[idx])
        # Aggiungere etichette sopra le barre se necessario
        for i, count in enumerate(sort_count):
            if sort_perc[i]!=0:
                # ax.text(posizione_barre[i], count, count, ha='center')
                ax.text(posizione_barre[i], count, str(round(sort_perc[i],2))+'%', ha='center')
                print(sort_types[i] + ' ' + str(sort_perc[i]))
                #pass
            else:
                ax.text(posizione_barre[i], count, '0%', ha='center')
                #pass
    
    # Rimuovere assi inutili in caso di subplot vuoti
    for ax in axes[len(types):]:
        ax.axis('off')
    
    # Titolo generale
    fig.suptitle(title, fontsize=16)
    
    # Salvataggio e visualizzazione
    plt.savefig(path, format='pdf', bbox_inches='tight')
    plt.close()


## Neurotrasmettitore tutti i neuroni

In [ ]:
types_nt_dict_count = {}
aree = list({attr.get('superclass', 'unknown') for _, attr in G.nodes(data=True)})
aree.remove('endocrine')
aree.remove('motor')
aree.remove('descending')
aree = aree + ['endocrine', 'motor', 'descending']

for area in aree:
    nodes_area = [nodo for nodo, attr in G.nodes(data=True) if attr.get('superclass') == area]
    nodes_nt_types = {key: 0 for key in ntransmitters}
    # nodes_nt_types['N/A'] = 0

    for node in nodes_area:
        n_nt_type = G.nodes[node].get('nt')
        if n_nt_type and not (isinstance(n_nt_type, float) and math.isnan(n_nt_type)):
            nodes_nt_types[n_nt_type] += 1
        else:
            # nodes_nt_types['N/A'] += 1
            pass

    types_nt_dict_count[area] = nodes_nt_types

In [ ]:
rows = []
for area, types in types_nt_dict_count.items():
    init = {key: 0 for key in ntransmitters}
    i_counts = Counter(types)
    counts = dict(Counter(init) + i_counts)
    counts = Counter({key: init.get(key, 0) + i_counts.get(key, 0) for key in set(init) | set(i_counts)})
    for ctype, count in counts.items():
        rows.append({"Area": area, "Type": ctype, "Count": count})

df = pd.DataFrame(rows)

In [ ]:
types = []
count = []

for i, area in enumerate(df["Area"].unique()):
    subset = df[df["Area"] == area]  
    subset = subset.sort_values(by="Count", ascending=False)
    types.append(subset["Type"])
    aree.append(subset["Area"].iloc[0])
    count.append(subset["Count"])

In [ ]:
grafico_a_barre_multi(aree, types, types_nt_dict_count, count, "All neurons", "AdultaImg/all_neurons_neurotransmitters.pdf", 1)

## Neurotrasmettitore colonist

In [ ]:
types_nt_dict_count_colonists = {}
aree = list({attr.get('superclass', 'unknown') for _, attr in G.nodes(data=True)})

for area in aree:
    colonists_area = colonists_per_area[area]
    nodes_nt_types = {key: 0 for key in ntransmitters}
    # nodes_nt_types['N/A'] = 0

    for node in colonists_area:
        n_nt_type = G.nodes[node].get('nt')
        if n_nt_type and not (isinstance(n_nt_type, float) and math.isnan(n_nt_type)):
            nodes_nt_types[n_nt_type] += 1
        else:
            # nodes_nt_types['N/A'] += 1
            pass
    if len(colonists_area):
        types_nt_dict_count_colonists[area] = nodes_nt_types

In [ ]:
rows = []
for area, types in types_nt_dict_count_colonists.items():
    init = {key: 0 for key in ntransmitters}
    i_counts = Counter(types)
    counts = dict(Counter(init) + i_counts)
    counts = Counter({key: init.get(key, 0) + i_counts.get(key, 0) for key in set(init) | set(i_counts)})
    for ctype, count in counts.items():
        rows.append({"Area": area, "Type": ctype, "Count": count})

df = pd.DataFrame(rows)

In [ ]:
types = []
count = []
aree = []

for i, area in enumerate(df["Area"].unique()):
    subset = df[df["Area"] == area]
    subset = subset.sort_values(by="Count", ascending=False)
    types.append(subset["Type"])
    aree.append(subset["Area"].iloc[0])
    count.append(subset["Count"])

In [ ]:
grafico_a_barre_multi(aree, types, types_nt_dict_count, count, "Colonists", "AdultaImg/colonists_neurotransmitters.pdf", 0)

# Connessioni tra emisferi

In [ ]:
superclasses = list({attr.get('superclass', 'unknown') for _, attr in G.nodes(data=True)})

In [ ]:
adulta_values = pd.DataFrame(columns=['area', 'colonists','s_colonists', 'd_colonists', 'destro', 'sinistro', 'destro_percentuale', 'sinistro_percentuale', 'list_destro', 'list_sinistro'])

for area in superclasses:
    nodes_left = [nodo for nodo, attr in G.nodes(data=True) if attr.get('superclass') == area and attr.get('hemisphere') == 'left']
    nodes_right = [nodo for nodo, attr in G.nodes(data=True) if attr.get('superclass') == area and attr.get('hemisphere') == 'right']
    colonists = set(nodes_left+nodes_right).intersection(colonists_nodes)
    d_colonists = set(nodes_right).intersection(colonists_nodes)
    s_colonists = set(nodes_left).intersection(colonists_nodes)
    destro = len(d_colonists)
    sinistro = len(s_colonists)
    destro_percentuale = destro/len(nodes_left)*100
    sinistro_percentuale = sinistro/len(nodes_right)*100
    
    info_dict = {
        'area': area,
        'colonists': colonists,
        's_colonists': s_colonists,
        'd_colonists': d_colonists,
        'destro': destro,
        'sinistro': sinistro,
        'destro_percentuale': destro_percentuale,
        'sinistro_percentuale': sinistro_percentuale,
        'list_destro': nodes_right,
        'list_sinistro': nodes_left
    }
    
    adulta_values = adulta_values._append(info_dict, ignore_index=True)

In [ ]:
adulta_values

In [ ]:
def get_graph_plot(igraph_graph, dataframe, ad_la, case=0):
    graph = nx.Graph()
    nodes = []
    for area in list(dataframe['area']):
        nodes.append((f"s_{area}", {"label": area}))
        nodes.append((f"d_{area}", {"label": area}))
        
    graph.add_nodes_from(nodes)
    node_sizes = []
    for row in dataframe.itertuples():
        if ad_la == "larva":
            node_sizes.append(len(row[3])*100)
            node_sizes.append(len(row[4])*100)
        else:
            node_sizes.append(row[6]*2)
            node_sizes.append(row[5]*2)
    # Define colors for each area
    colors_dict = {
        's_sensory': '#FFB6C1',
        's_ascending': '#FF6347',
        's_sensory_l': '#FFB6C1',
        's_ascending_l': '#FF6347',
        's_innate': '#90EE90',
        's_endocrine': '#90EE90',
        's_learning/memory': "#87CEFA",
        's_motor': "#87CEFA",
        's_deep brain': "#FFFE02",
        's_central': "#FFFE02",
        's_pre-output': "#6495ED",
        's_optic': "#6495ED",
        's_brain outputs': "#F4A460",
        's_descending': "#F4A460",
        'd_sensory': '#FFB6C1',
        'd_ascending': '#FF6347',
        'd_sensory_l': '#FFB6C1',
        'd_ascending_l': '#FF6347',
        'd_innate': '#90EE90',
        'd_endocrine': '#90EE90',
        'd_learning/memory': "#87CEFA",
        'd_motor': "#87CEFA",
        'd_deep brain': "#FFFE02",
        'd_central': "#FFFE02",
        'd_descending': "#F4A460",
        'd_pre-output': "#6495ED",
        'd_optic': "#6495ED",
        'd_brain outputs': "#F4A460",
        's_visual_projection': "#D2B48C",
        'd_visual_projection': "#D2B48C",
        's_visual_centrifugal': "#DA70D6",
        'd_visual_centrifugal': "#DA70D6",
    }
    
    # Apply colors based on the custom order
    colors = [colors_dict[area[0]] for area in nodes]

    nodes_areas = {}
    powernodes = []
    for row in dataframe.itertuples():
        area = str(row[1])
        destro = list(row[9])
        sinistro = list(row[10])
        for n in destro:
            nodes_areas[n] = "d_"+area
        for n in sinistro:
            nodes_areas[n] = "s_"+area
        powernodes.extend(list(row[2]))
        
    edges_larva = [(str(igraph_graph.vs.find(e.source)['_nx_name']), str(igraph_graph.vs.find(e.target)['_nx_name'])) for e in list(igraph_graph.es)]
    
    edges_areas = []
    for node1, node2 in edges_larva:
        try:
            to_add = (nodes_areas[node1], nodes_areas[node2])
            if case == 0:
                if node1 in powernodes or node2 in powernodes:
                    edges_areas.append(to_add)
            elif case == 1:
                if node1 in powernodes and node2 in powernodes:
                    edges_areas.append(to_add)
        except:
            continue
    
    edge_dict={}
    for edge in edges_areas:
        if edge in edge_dict:
            edge_dict[edge]+=1
        else:
            edge_dict[edge]=1
    
    edges_areas = []
    for edge in edge_dict:
        if edge[0] != edge[1]:
            edges_areas.append((edge[0], edge[1], edge_dict[edge]))

    #print(edges_areas)
    with open('Adulta/edges_areas_entrambi.pickle', 'wb') as f:
        pickle.dump(edges_areas, f)
    
    graph.add_weighted_edges_from(edges_areas)
    edge_weights = []
    for u, v in graph.edges():
        weight = graph[u][v]['weight']
        edge_weights.append(weight)
    max_w = max(edge_weights)
    min_w = min(edge_weights)
    print(min_w)
    print(max_w)
    print(edge_weights)
    
    edge_weights_app = edge_weights
    edge_weights = []
    for weight in edge_weights_app:
        if weight<10:
            edge_weights.append(0.1)
        elif weight<100:
            edge_weights.append(0.4)
        elif weight<500:
            edge_weights.append(0.8)
        elif weight<1000:
            edge_weights.append(1)
        elif weight<5000:
            edge_weights.append(3)
        elif weight<10000:
            edge_weights.append(5)
        else:
            edge_weights.append(10)
    
    # edge_weights = [((w-min_w)/(max_w-min_w))*20 for w in edge_weights]
    node_labels = {node: graph.nodes[node]['label'].replace("_l", "").replace("_", " ") for node in graph.nodes()}

    pos = {
        "s_pre-output": (5,0),
        "s_optic": (0,7),
        "s_deep brain": (0,2),
        "s_central": (4,6),
        "s_learning/memory": (3,5),
        "s_motor": (3,1),
        'd_learning/memory': (3,12),
        'd_motor': (9,1),
        'd_deep brain': (0,15),
        'd_central': (8,6),
        'd_pre-output': (5,17),
        'd_optic': (12,7),
        'd_innate': (7,12),
        'd_endocrine': (10,2),
        "s_innate": (7,5),
        "s_endocrine": (2,2),
        "s_visual_centrifugal": (1,5),
        "s_visual_projection": (1,9),
        "s_brain outputs": (9,8),
        "d_visual_centrifugal": (11,5),
        "d_visual_projection": (11,9),
        "s_descending": (5,0),
        'd_brain outputs': (9,10),
        'd_descending': (7,0),
        "s_ascending": (5,12),
        "d_ascending": (7,12),
        "s_sensory": (3,11),
        "d_sensory": (9,11),
        "s_sensory_l": (17,6),
        "d_sensory_l": (17,11),
        "s_ascending_l": (15,4),
        "d_ascending_l": (15,13),
    }



    # Create the plot
    fig, ax = plt.subplots(figsize=(18, 18))
    # Set the background to be transparent
    fig.patch.set_alpha(0.0)
    ax.set_facecolor('none')
    
    # Draw the graph with custom positions and properties
    nx.draw(
        graph, pos, with_labels=True, labels=node_labels,
        node_size=node_sizes, node_color=colors,
        edge_color='gray', font_size=18, font_color='black',
        width=edge_weights, connectionstyle='arc3,rad=0.2',
        ax=ax
    )

    legend_elements = [
        Line2D([0], [0], color='grey', lw=0.1, label='weight < 10'),
        Line2D([0], [0], color='grey', lw=0.4, label='10 ≤ weight < 100'),
        Line2D([0], [0], color='grey', lw=0.8, label='100 ≤ weight < 500'),
        Line2D([0], [0], color='grey', lw=1, label='100 ≤ weight < 1,000'),
        Line2D([0], [0], color='grey', lw=3, label='1,000 ≤ weight < 5,000'),
        Line2D([0], [0], color='grey', lw=5, label='5,000 ≤ weight < 10,000'),
        Line2D([0], [0], color='grey', lw=10, label='weight ≥ 10,000')
    ]

    # ax.legend(handles=legend_elements, loc='upper left', fontsize=30)
    # Aggiungi la legenda sopra la figura
    ax.legend(
        handles=legend_elements, loc='lower left', fontsize=18, ncol=4,
        bbox_to_anchor=(0, 0.9),  # Sopra il grafico
        frameon=False  # Rimuove il riquadro della legenda
    )
    
    # Adjust margins to avoid cutting off labels
    ax.margins(0.20)
    plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
    
    # Save and show the plot
    plt.savefig(f'AdultaImg/graph_output_{ad_la}_{case+1}.png', facecolor=fig.get_facecolor())
    plt.show()

In [ ]:
adulta_values

In [ ]:
get_graph_plot(G_igraph, adulta_values, "adulta", 0)

In [ ]:
get_graph_plot(G_igraph, adulta_values, "adulta", 1)

In [ ]:
adulta_values

In [ ]:
with open('Adulta/edges_areas_unosolo.pickle', 'rb') as f:
    edges_areas_unosolo = pickle.load(f)

In [ ]:
edges_areas_unosolo = sorted(edges_areas_unosolo, key=lambda x: x[2], reverse=True)

In [ ]:
edges_areas_unosolo

In [ ]:
with open('Adulta/edges_areas_entrambi.pickle', 'rb') as f:
    edges_areas_entrambi = pickle.load(f)

In [ ]:
edges_areas_entrambi = sorted(edges_areas_entrambi, key=lambda x: x[2], reverse=True)

In [ ]:
edges_areas_entrambi

# Connessioni inter o intra emisfero

In [ ]:
with open('Adulta/colonists_2.5.pickle', 'rb') as f:
        colonists = pickle.load(f)

In [ ]:
inter_intra_connections_nodes = {'intra':[], 'inter':[], 'both':[]}
inter_intra_numbers = {'only intra':0, 'only inter':0, 'both':0}
no_hemisphere = []

for colonist, targets in colonists.items():
    superclass = G.nodes[colonist]['superclass']
    colonist_hemisfere = G.nodes[colonist]['hemisphere']
    colonist_connections = {'intra':False, 'inter':False}
    flat_targets = [
        x
        for xs in targets
        for x in xs
    ]
    if colonist_hemisfere!=colonist_hemisfere:
        no_hemisphere.append(colonist)
        continue
    else:
       for target in flat_targets:
            target_hemisfere = G.nodes[target]['hemisphere']
            if colonist_hemisfere==target_hemisfere:
                colonist_connections['intra']=True
            else:
                colonist_connections['inter']=True
    if colonist_connections['intra'] and colonist_connections['inter']:
        inter_intra_connections_nodes['both'].append(colonist)
        inter_intra_numbers['both']+=1
    elif colonist_connections['intra']:
        inter_intra_connections_nodes['intra'].append(colonist)
        inter_intra_numbers['only intra']+=1
    else:
        inter_intra_connections_nodes['inter'].append(colonist)
        inter_intra_numbers['only inter']+=1
            

In [ ]:
hemisphere_attributes = {G.nodes[node]['hemisphere'] for node in colonists}
print(hemisphere_attributes)

In [ ]:
inter_intra_numbers

In [ ]:
areas_inter_intra_connections_nodes = {key:[] for key in superclasses}

for colonist in inter_intra_connections_nodes['both']:
    superclass = G.nodes[colonist]['superclass']
    areas_inter_intra_connections_nodes[superclass].append(colonist)

In [ ]:
areas_intra_connections_nodes = {key:[] for key in superclasses}

for colonist in inter_intra_connections_nodes['intra']:
    superclass = G.nodes[colonist]['superclass']
    areas_intra_connections_nodes[superclass].append(colonist)

In [ ]:
areas_inter_connections_nodes = {key:[] for key in superclasses}

for colonist in inter_intra_connections_nodes['inter']:
    superclass = G.nodes[colonist]['superclass']
    areas_inter_connections_nodes[superclass].append(colonist)

In [ ]:
areas_intra_connections_nodes['visual_projection']

In [ ]:
colonist = '720575940611891437'
targets = colonists[colonist]
superclass = G.nodes[colonist]['superclass']
colonist_hemisfere = G.nodes[colonist]['hemisphere']
colonist_connections = {'intra':False, 'inter':False}

flat_targets = [
    x
    for xs in targets
    for x in xs
]
for target in flat_targets:
    target_hemisfere = G.nodes[target]['hemisphere']
    if colonist_hemisfere==target_hemisfere:
        print('intra ' + target_hemisfere + ' ' + target)
    else:
        print('inter ' + target_hemisfere + ' ' + target)

            

# Connessioni inter o intra emisfero

In [ ]:
with open('Adulta/dist_info_superclasses.pickle', 'rb') as f:
    dist_info_superclasses = pickle.load(f)

In [ ]:
inter_intra_connections = {}
inter_intra_connections_nodes = {'intra':[], 'inter':[], 'both':[]}
inter_intra_numbers = {'only intra':0, 'only inter':0, 'both':0}

for colonist in colonists_nodes:
    hemisfere = G.nodes[colonist]['hemisphere']
    superclass = G.nodes[colonist]['superclass']
    inter_intra_connections[colonist] = {
        'inter':0,
        'intra':0
    }
    nodes_connections = {'inter': False, 'intra':False}
    out_edges = list(G.out_edges(colonist, data=True))
    for edge in out_edges:
        if edge[2]['weight'] > 2.5*dist_info_superclasses[superclass]['average']:
            if G.nodes[edge[1]]['hemisphere']==hemisfere:
                inter_intra_connections[colonist]['intra']+=1
                nodes_connections['intra'] = True
            else:
                inter_intra_connections[colonist]['inter']+=1
                nodes_connections['inter'] = True
    
    if nodes_connections['inter'] and nodes_connections['intra']:
        inter_intra_connections_nodes['both'].append(colonist)
        inter_intra_numbers['both']+=1
    elif nodes_connections['inter']:
        inter_intra_connections_nodes['inter'].append(colonist)
        inter_intra_numbers['only inter']+=1
    else:
        inter_intra_connections_nodes['intra'].append(colonist)
        inter_intra_numbers['only intra']+=1

In [ ]:
inter_intra_numbers

In [ ]:
areas_inter_intra_connections_nodes = {key:[] for key in superclasses}

for colonist in inter_intra_connections_nodes['both']:
    superclass = G.nodes[colonist]['superclass']
    areas_inter_intra_connections_nodes[superclass].append(colonist) 

In [ ]:
areas_intra_connections_nodes = {key:[] for key in superclasses}

for colonist in inter_intra_connections_nodes['intra']:
    superclass = G.nodes[colonist]['superclass']
    areas_intra_connections_nodes[superclass].append(colonist)

In [ ]:
areas_intra_connections_nodes = {key:[] for key in superclasses}

for colonist in inter_intra_connections_nodes['intra']:
    superclass = G.nodes[colonist]['superclass']
    areas_intra_connections_nodes[superclass].append(colonist)

In [ ]:
colonist = '720575940620538806'

hemisfere = G.nodes[colonist]['hemisphere']
superclass = G.nodes[colonist]['superclass']
inter_intra_connections[colonist] = {
    'inter':0,
    'intra':0
}
nodes_connections = {'inter': False, 'intra':False}
out_edges = list(G.out_edges(colonist, data=True))
for edge in out_edges:
    if edge[2]['weight'] > 2.5*dist_info_superclasses[superclass]['average']:
        if G.nodes[edge[1]]['hemisphere']==hemisfere:
            print('intra' + edge[1])
        else:
            print('inter' + edge[1])

In [ ]:
with open('Adulta/dist_info_superclasses.pickle', 'rb') as f:
    dist_info_superclasses = pickle.load(f)

In [ ]:
for celltype, info in dist_info_superclasses.items():
    print(celltype + ' average distance: ' + str(round(info['average'],4)))

In [ ]:
degree_connectivity = list(nx.average_degree_connectivity(G).values())
sum(degree_connectivity)/len(degree_connectivity)

In [ ]:
degree_connectivity = list(nx.average_degree_connectivity(G, weight='weight').values())
sum(degree_connectivity)/len(degree_connectivity)


In [ ]:
G.nodes['720575940626979621']['y']

In [ ]:
pesi = []

for u, v, data in G.edges(data=True):
    pesi.append(data['weight'])

max(pesi)

# Backbone

In [ ]:
def compute_measures(graph):
    graph_igraph = ig.Graph.from_networkx(graph)
    
    print('Number of nodes: ' + str(graph_igraph.vcount()))
    print('Number of edges: ' + str(graph_igraph.ecount()))
    
    in_degrees = [deg for _, deg in graph.in_degree()]  # Lista degli in-degree per ciascun nodo
    average_in_degree = sum(in_degrees) / len(graph.nodes())
    print(f"Average node in-degree: {average_in_degree}")

    out_degrees = [deg for _, deg in graph.out_degree()]  # Lista degli in-degree per ciascun nodo
    average_out_degree = sum(out_degrees) / len(graph.nodes())
    print(f"Average node out-degree: {average_in_degree}")

    weighted_in_degrees = [deg for _, deg in graph.in_degree(weight='weight')]  # In-degree pesato
    average_weighted_in_degree = sum(weighted_in_degrees) / len(graph.nodes())
    print(f"Average Weighted In-Degree: {average_weighted_in_degree}")

    weighted_out_degrees = [deg for _, deg in graph.out_degree(weight='weight')]  # In-degree pesato
    average_weighted_out_degree = sum(weighted_out_degrees) / len(graph.nodes())
    print(f"Average Weighted Out-Degree: {average_weighted_out_degree}")
    
    print('Density: ' + str(graph_igraph.density()))
    
    #G_undirected = graph.to_undirected()
    #avg_clustering = nx.average_clustering(G_undirected)
    #avg_clustering = graph_igraph.transitivity_avglocal_undirected()
    #avg_clustering = graph_igraph.transitivity_undirected()
    print('Average Clustering Coefficient: ' + str(nx.average_clustering(graph)))
    
    print('Average path length C: ' + str(graph_igraph.average_path_length()))
    print('Diameter: ' + str(graph_igraph.diameter()))
    print('Degree Assortativity: ' + str(graph_igraph.assortativity_degree()))

    largest_component = graph_igraph.clusters().giant() 
    print('Maximum (strongly) Connected Component: ' + str(largest_component.vcount()))


In [ ]:
compute_measures(G)

In [ ]:
G_colonists = nx.induced_subgraph(G, colonists_nodes)

In [ ]:
compute_measures(G_colonists)